In [1]:
!pip install tensorflow

In [ ]:
# Neural Collaborative Filtering (NCF) with MovieLens 1M
# !pip install recommenders scikit-learn pandas numpy

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from recommenders.datasets import movielens
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.utils.constants import SEED
from recommenders.evaluation.python_evaluation import map_at_k, ndcg_at_k, precision_at_k, recall_at_k
from recommenders.utils.timer import Timer
from recommenders.datasets.python_splitters import python_chrono_split
from recommenders.evaluation.python_evaluation import get_top_k_items

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import os
import tempfile

print("Downloading MovieLens 1M dataset...")
df = movielens.load_pandas_df(size="1m")  # columns: userID, itemID, rating

print("Data preview:")
print(df.head())

# Rename columns to expected format
ratings = df.rename(columns={"userID": "userID", "itemID": "itemID", "rating": "rating"})

print("Splitting data into train and test sets...")
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=SEED)
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# Sort by userID as required by NCFDataset
train_df = train_df.sort_values(by="userID")
test_df = test_df.sort_values(by="userID")

# Save temporary CSVs as required by NCFDataset
with tempfile.TemporaryDirectory() as temp_dir:
    train_path = os.path.join(temp_dir, "train.csv")
    test_path = os.path.join(temp_dir, "test.csv")
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Converting to NCFDataset...")
    data = NCFDataset(train_file=train_path, test_file=test_path, seed=SEED)

    print("Initializing NCF model...")
    ncf_model = NCF(
        n_users=data.n_users,
        n_items=data.n_items,
        model_type="NeuMF",  # Options: "NeuMF", "GMF", "MLP"
        learning_rate=0.001,
        batch_size=256,
        verbose=10,
        seed=SEED,
    )

    print("Training the model...")
    with Timer() as train_time:
        ncf_model.fit(data)

    print(f"Training took: {train_time}")

    print("Generating top-K predictions...")
    top_k = 10

    # Only use users and items seen during training
    known_users = set(train_df["userID"].unique())
    known_items = set(train_df["itemID"].unique())

    test_users = test_df[test_df.userID.isin(known_users)].userID.unique()
    all_items = df[df.itemID.isin(known_items)].itemID.unique()

    user_input, item_input = [], []
    for u in test_users:
        user_input.extend([u] * len(all_items))
        item_input.extend(all_items)

    scores = ncf_model.predict(user_input, item_input, is_list=True)
    all_preds = pd.DataFrame({"userID": user_input, "itemID": item_input, "prediction": scores})
    all_preds.rename(columns={"prediction": "rating"}, inplace=True)
    top_k_preds = get_top_k_items(all_preds, k=top_k)
    top_k_preds["prediction"] = top_k_preds["rating"]  # copy predicted score
    top_k_preds["rating"] = 1  # dummy relevancy score
    top_k_preds = top_k_preds[["userID", "itemID", "rating", "prediction"]].astype({
        "userID": int, "itemID": int, "rating": int, "prediction": float
    })

    test_df_filtered = test_df[test_df.userID.isin(test_users) & test_df.itemID.isin(known_items)].copy()
    test_df_filtered = test_df_filtered[["userID", "itemID", "rating"]].astype({
        "userID": int, "itemID": int, "rating": int
    })

    print("Evaluating model (Top-K metrics)...")
    k = 10
    print(f"MAP@{k}: {map_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")
    print(f"NDCG@{k}: {ndcg_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")
    print(f"Precision@{k}: {precision_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")
    print(f"Recall@{k}: {recall_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=k):.4f}")





INFO:recommenders.datasets.download_utils:Downloading https://files.grouplens.org/datasets/movielens/ml-1m.zip
100%|██████████| 5.78k/5.78k [00:01<00:00, 4.20kKB/s]


Data preview:
   userID  itemID  rating  timestamp
0       1    1193     5.0  978300760
1       1     661     3.0  978302109
2       1     914     3.0  978301968
3       1    3408     4.0  978300275
4       1    2355     5.0  978824291
Splitting data into train and test sets...
Train size: 800167, Test size: 200042


INFO:recommenders.models.ncf.dataset:Indexing /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmp85ehih6j/train.csv ...


Converting to NCFDataset...


INFO:recommenders.models.ncf.dataset:Indexing /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmp85ehih6j/test.csv ...
INFO:recommenders.models.ncf.dataset:Creating full leave-one-out test file /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmp85ehih6j/test_full.csv ...
100%|██████████| 6038/6038 [01:21<00:00, 74.46it/s] 
INFO:recommenders.models.ncf.dataset:Indexing /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmp85ehih6j/test_full.csv ...


Initializing NCF model...


/opt/anaconda3/envs/recommenders/lib/python3.9/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Training the model...


INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [21.66s]: train_loss = 0.278247 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 20 [22.54s]: train_loss = 0.273462 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 30 [21.82s]: train_loss = 0.270608 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 40 [21.22s]: train_loss = 0.268691 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 50 [22.41s]: train_loss = 0.267076 


Training took: 1098.5637
Generating top-K predictions...
Evaluating model (Top-K metrics)...
MAP@10: 0.0558
NDCG@10: 0.1354
Precision@10: 0.1248
Recall@10: 0.0728


In [40]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.utils.constants import SEED
from recommenders.evaluation.python_evaluation import map_at_k, ndcg_at_k, precision_at_k, recall_at_k, get_top_k_items
from recommenders.utils.timer import Timer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import os
import tempfile

# === Step 1: Load and preprocess your CSV ===
csv_path = "rating-South_Carolina.csv"  # ⬅️ Update this with your path
df_raw = pd.read_csv(csv_path)

# Split 'business' column into itemID and userID
df_split = df_raw['business'].str.split(':', expand=True)
df = pd.DataFrame({
    'itemID': df_split[0],
    'userID': df_split[1],
    'rating': df_raw['rating']
})

# Encode user and item IDs as integers
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df['userID'] = user_encoder.fit_transform(df['userID'])
df['itemID'] = item_encoder.fit_transform(df['itemID'])
df['rating'] = df['rating'].astype(float)

print("Data preview:")
print(df.head())

# === Step 2: Train/test split ===
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)
train_df = train_df.sort_values("userID")
test_df = test_df.sort_values("userID")

# === Step 3: Save CSVs for NCF ===
with tempfile.TemporaryDirectory() as temp_dir:
    train_path = os.path.join(temp_dir, "train.csv")
    test_path = os.path.join(temp_dir, "test.csv")
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    data = NCFDataset(train_file=train_path, test_file=test_path, seed=SEED)

    # === Step 4: Train NCF model ===
    ncf_model = NCF(
        n_users=data.n_users,
        n_items=data.n_items,
        model_type="NeuMF",
        learning_rate=0.001,
        batch_size=256,
        verbose=1,
        seed=SEED,
    )

    print("Training the model...")
    with Timer() as train_time:
        ncf_model.fit(data)
    print(f"Training took: {train_time}")

    # === Step 5: Generate predictions ===
    top_k = 10
    known_users = set(train_df["userID"].unique())
    known_items = set(train_df["itemID"].unique())

    test_users = test_df[test_df.userID.isin(known_users)].userID.unique()
    all_items = df[df.itemID.isin(known_items)].itemID.unique()

    user_input, item_input = [], []
    for u in test_users:
        user_input.extend([u] * len(all_items))
        item_input.extend(all_items)

    print("Predicting scores...")
    scores = ncf_model.predict(user_input, item_input, is_list=True)
    all_preds = pd.DataFrame({"userID": user_input, "itemID": item_input, "prediction": scores})
    all_preds.rename(columns={"prediction": "rating"}, inplace=True)

    # === Step 6: Top-K predictions ===
    top_k_preds = get_top_k_items(all_preds, k=top_k)
    top_k_preds["prediction"] = top_k_preds["rating"]
    top_k_preds["rating"] = 1  # dummy relevance
    top_k_preds = top_k_preds[["userID", "itemID", "rating", "prediction"]].astype({
        "userID": int, "itemID": int, "rating": int, "prediction": float
    })

    test_df_filtered = test_df[test_df.userID.isin(test_users) & test_df.itemID.isin(known_items)].copy()
    test_df_filtered = test_df_filtered[["userID", "itemID", "rating"]].astype({
        "userID": int, "itemID": int, "rating": int
    })

    # === Step 7: Evaluation ===
    print("Evaluating model (Top-K metrics)...")
    print(f"MAP@{top_k}: {map_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=top_k):.4f}")
    print(f"NDCG@{top_k}: {ndcg_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=top_k):.4f}")
    print(f"Precision@{top_k}: {precision_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=top_k):.4f}")
    print(f"Recall@{top_k}: {recall_at_k(test_df_filtered, top_k_preds, col_prediction='prediction', k=top_k):.4f}")


Data preview:
   itemID  userID  rating
0   66785   22330     4.0
1   66785   22330     5.0
2    9491   57452     4.0
3    9491   57452     5.0
4    9491   57452     4.0


INFO:recommenders.models.ncf.dataset:Indexing /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmpe4dlvtqw/train.csv ...
INFO:recommenders.models.ncf.dataset:Indexing /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmpe4dlvtqw/test.csv ...
INFO:recommenders.models.ncf.dataset:Creating full leave-one-out test file /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmpe4dlvtqw/test_full.csv ...
100%|██████████| 75036/75036 [41:10<00:00, 30.37it/s]  
INFO:recommenders.models.ncf.dataset:Indexing /var/folders/np/xm23r59s3_7gc35f52rnpczm0000gn/T/tmpe4dlvtqw/test_full.csv ...
/opt/anaconda3/envs/recommenders/lib/python3.9/site-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


Training the model...


INFO:recommenders.models.ncf.ncf_singlenode:Epoch 1 [4131.68s]: train_loss = 0.047591 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 2 [4249.85s]: train_loss = 0.015819 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 3 [3794.74s]: train_loss = 0.011363 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 4 [3809.61s]: train_loss = 0.009609 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 5 [3850.17s]: train_loss = 0.008489 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 6 [3827.43s]: train_loss = 0.007958 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 7 [3838.59s]: train_loss = 0.007631 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 8 [3862.94s]: train_loss = 0.007298 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 9 [3938.12s]: train_loss = 0.007034 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 10 [4037.70s]: train_loss = 0.006501 
INFO:recommenders.models.ncf.ncf_singlenode:Epoch 11 [4099.27s]: train_loss = 0.005702 
INFO:recommenders.models.ncf.ncf_singleno

KeyboardInterrupt: 